In [1]:
from __future__ import annotations

import numpy as np
import netCDF4
import yaml
import xarray as xr
from modular_dales import dales_simulation
from modular_dales.Atmosphere import (
    AtmosphereModule,
    AtmosphericProfile,
    InterpolatedProfile,
)
import subprocess
from modular_dales.Configuration.defaultnamelist import DefaultNamelistModule
from modular_dales.Configuration.output_modules import EasyOutputModule
from modular_dales.Configuration.run_and_time import TimeModule
from modular_dales.Geometry.GridDales import GridDales
from modular_dales.modular.time_dependent import (
    TimeDependentScalar,
    TimedependentModule,
)
from modular_dales.Surface.surface import ConstantSurfaceTemperatureModule
from modular_dales.IBM import IBMModule, FromAHN
from modular_dales.vars import *  # noqa: F401,F403

with open("machine_conf.yaml", "r") as file:
    machine_conf = yaml.safe_load(file)

Roughness element should be at least 1m for MOST stability!!! Setting to 1m
Roughness element should be at least 1m for MOST stability!!! Setting to 1m
Roughness element should be at least 1m for MOST stability!!! Setting to 1m


In [56]:
"""Construct a simulation with time-dependent scalar params inside AtmosphericProfile."""
sim = dales_simulation("timedep_atmosphere_params", machine_conf)
sim += DefaultNamelistModule()
domain_info = GridDales(
    itot=8,
    jtot=8,
    kmax=130,
    xsize=160.0,
    ysize=160.0,
    kmax_soil=4,
    xlat=52.25,
    xlon=5.45,
    x0=137167.763496,
    y0=454496.628974,
    alpha=1.02,
    dz0=20,
    proj4="+proj=sterea +lat_0=52.1561605555556 +lon_0=5.38763888888889 +k=0.9999079 +x_0=155000 +y_0=463000 +ellps=bessel +towgs84=565.4171,50.3319,465.5524,1.9342,-1.6677,9.1019,4.0725 +units=m +no_defs +type=crs",
)
sim += domain_info


timesteps = [0,3600,6700, 7200,7800,3600*3]
sim += TimedependentModule(timesteps=timesteps)


atmo = AtmosphereModule()

atmo += AtmosphericProfile(
    variable=ua, shape="lin", params=dict(surf_val=1, ddz=1e-3)
)
atmo += AtmosphericProfile(
    variable=va, shape="lin", params=dict(surf_val=0, ddz=0))

# atmo += AtmosphericProfile(
#     variable=thetal, shape="lin", params=dict(surf_val=293.15, ddz=5e-3)
# )
atmo += InterpolatedProfile(
    variable=thetal,
    z=[0, 1000, 2000, float(domain_info.zt.max())],
    points=[293.15, 293.15, 294.15, 298.15],
)
# atmo += AtmosphericProfile(
#     variable=qt, shape="lin", params=dict(surf_val=0, ddz=0))
atmo += InterpolatedProfile(
    variable=qt,
    z=[0, 1000, 2000, 4000, float(domain_info.zt.max())],
    points=[0.0018, 0.0018, 0.0013, 0, 0.0000],
)
# atmo += InterpolatedProfile(
#     variable=tnqt_adv,
#     z=[0, 350, 400, 450, float(domain_info.zt.max())],
#     points=[0, 0, TimeDependentScalar(times=timesteps,values=[0,0,0,1e-3,0,0]),0, 0]
# )
atmo += InterpolatedProfile(
    variable=tke,
    z=[0, 4000, 5000],
    points=[0, 0, 0],
)
sim += atmo


sim += ConstantSurfaceTemperatureModule(
    thls=TimeDependentScalar(times=timesteps, values=[293.15,293.15,299.15,293.15,293.15,293.15]),
    z0mav=0.0001,
    z0hav=0.0001,
    ps=100000,
    albedoav=0.22,
)

# ibm = IBMModule()
# ibm += FromAHN()
# sim += ibm

                  


sim += TimeModule(xtime=0.0, xday=1, xyear=2025, runtime=3600*3)
output_dict = {
                    "namfielddump:":60,
                "namcape":10,
                "namlsmcrosssection:dtav":60,
                "namgenstat:dtav":10,
                "namcrosssection:dtav":10,
                "namtimestat:dtav":10,
                "nambudget:dtav":10,
                "nambudget:timeav":10,
                "namgenstat:timeav":10,
}
sim += EasyOutputModule(output_interval=10)

if sim.nml.get("namchecksim") is None:
    sim.nml["namchecksim"] = {}
sim.nml["namchecksim"]["tcheck"] = 360*2
if sim.nml.get("namnetcdfstats") is None:
    sim.nml["namnetcdfstats"] = {}
sim.nml["namnetcdfstats"]["lsync"] = True
if sim.nml.get("nammicrophysics") is None:
    sim.nml["nammicrophysics"] = {}
sim.nml["nammicrophysics"]["imicro"] = 2

sim.sim_preprocessing_pipeline()
print(sim.output_path)

No job_template_path specified in machine configuration; using default template input_template/job.001


/Users/andrevanginkel/Documents/40_Input_and_Runs/42_Dales_Cases/42.01_generated_cases/timedep_atmosphere_params


In [57]:
ds = None

subprocess.run(["rm","-rf", "run_001"],cwd=sim.output_path.as_posix())
subprocess.run(["rm","fielddump.nc","cape.nc","surfcross.nc"],cwd=sim.output_path.as_posix())
subprocess.run("./job.001", cwd=sim.output_path.as_posix())

subprocess.run(
    ["combine.sh", "run_001"], check=False, cwd=sim.output_path.as_posix())


Cold start
Transfer starting: 1 files
dales

sent 7489057 bytes  received 42 bytes  226256767 bytes/sec
total size is 7488056  speedup is 1.00
 DALES 5.0.0 git: v5.0.0-beta.1-93-ga4bd72-dirty
 namoptions.001                                                                                                                                                                                                                                                  
 MPI mesh nprocx, nprocy:            2           2
 e12 value is zero (or less) in prof.inp


    WARNING PE:     0 modglobal/initglobal: WARNING, You are working with a non-equidistant grid!!!!
    WARNING PE:     0 modtracers/inittracers: tracers.001.nc not found
    WARNING PE:     3 modtracers/inittracers: tracers.001.nc not found
    WARNING PE:     2 modtracers/inittracers: tracers.001.nc not found
    WARNING PE:     1 modtracers/inittracers: tracers.001.nc not found
 modstartup/checkinitvalues: jmax = jtot / nprocy = 4
 modstartup/checkinitvalues: imax = itot / nprocx = 4


Time of Day: 23:27:10.278 Time of Simulation:      720.00    dt:    2.2360
ETA:   00:00:20  8.04 sim_min/s   Scaling:   4.49E+05 (it/s)(gridpoints/cores)
Courant numbers (x,y,z,tot):  7.86E-01  6.95E-02  4.01E-02   14  7.89E-01  101
Cell Peclet number:  1.86E-03   15
divmax, divtot =    1.95E-08  -1.44E-04       dt limited by dt_lim         
Time of Day: 23:27:11.772 Time of Simulation:     1440.00    dt:    2.5000
ETA:   00:00:19  8.03 sim_min/s   Scaling:   4.01E+05 (it/s)(gridpoints/cores)
Courant numbers (x,y,z,tot):  8.56E-01  1.48E-01  2.12E-02    9  8.68E-01   99
Cell Peclet number:  1.51E-03   14
divmax, divtot =    2.09E-08   2.46E-05       dt limited by dt_lim         
Time of Day: 23:27:13.433 Time of Simulation:     2160.00    dt:    2.5000
ETA:   00:00:19  7.22 sim_min/s   Scaling:   3.61E+05 (it/s)(gridpoints/cores)
Courant numbers (x,y,z,tot):  8.32E-01  2.18E-01  1.47E-02   10  8.58E-01   99
Cell Peclet number:  1.05E-03    9
divmax, divtot =    2.20E-08   7.64E-05     

CompletedProcess(args=['combine.sh', 'run_001'], returncode=0)

In [95]:
import panel as pn
import hvplot.xarray  # noqa: F401
import pathlib
import dataclasses

@dataclasses.dataclass
class test:
    output_path: str
    
sim = test("")
sim.output_path = pathlib.Path("/private/var/folders/4j/9z5s90bd4r90cr7cd8rkm2j8k57wq5/T/pytest-of-andrevanginkel/pytest-30/test_ls2d_atmosphere_full_ls2d0/case_ls2d_atmosphere/ls2d_atmosphere")



# ds_fielddump.thl.mean(dim=("xt","yt")).plot(x="time")

In [96]:
# ds_fielddump.w.std(dim=("xt","yt")).plot(x="time")

In [97]:
output_path = sim.output_path
ds_fielddump = xr.open_dataset(output_path / "fielddump.nc") 
ds_tm = xr.open_dataset(output_path/"run_001"/"profiles.001.nc")
ds_cape = xr.open_dataset(output_path / "cape.nc")
ds_surfcross = xr.open_dataset(output_path / "surfcross.nc")
ds_tmser = xr.open_dataset(output_path / "run_001"/"tmser.001.nc")

In [135]:
import holoviews as hv
import hvplot.xarray  # noqa

hv.extension("bokeh")
def profiles():
    ds_sel = ds_tm.isel(time=slice(5,None))
    ln = len(ds_sel.zt)
    return ds_sel.interp(zt=np.linspace(ds_sel.zt.min().values,ds_sel.zt.max().values,num=ln),zm=np.linspace(ds_sel.zm.min().values,ds_sel.zm.max().values,num=ln))
def cape():
    ds = ds_cape
    ds_mean = ds_cape.mean(dim=("xt","yt"),keep_attrs=True)
    ds_std = ds_cape.std(dim=("xt","yt"),keep_attrs=True)
    return ds_mean, ds_std
def crosses():
    ds_mean = ds_surfcross.mean(dim=("xt","yt"),keep_attrs=True)
    ds_std = ds_surfcross.std(dim=("xt","yt"),keep_attrs=True)
    return ds_mean, ds_std
def get_label_to_var(ds):
    # Build label mapping: "short: long" -> short
    label_to_var = {}
    
    for v in ds.data_vars:
        long_name = ds[v].attrs.get("long_name", "")
        if long_name:
            label = f"{v}: {long_name}"
        else:
            label = v
        label_to_var[label] = v
    return label_to_var
def get_variable_dim(ds):
    return hv.Dimension(
    "variable",
    values=list(get_label_to_var(ds).keys()),
    label="Variable",
)

def plot_variable_heat(label, label_to_var, ds1, ds2=None):
    var = label_to_var[label]
    da = ds1[var]

    # detect vertical coordinate automatically
    if "zt" in da.dims:
        ydim = "zt"
    elif "zm" in da.dims:
        ydim = "zm"
    else:
        raise ValueError(f"{var} has no vertical dimension")
    # compute min/max safely (lazy compatible)
    vmin = float(da.min())
    vmax = float(da.max())
    # decide symmetric scaling
    if vmin < 0 and vmax > 0:
        vmax_abs = max(abs(vmin), abs(vmax))
        clim = (-vmax_abs, vmax_abs)
        cmap = "RdBu_r"   # diverging colormap
    else:
        clim = (vmin, vmax)
        cmap = "viridis"  # sequential colormap

    return da.hvplot(
        x="time",
        y=ydim,
        cmap=cmap,
        clim=clim,
        colorbar=True,
        title=label,
    ).opts(
    backend_opts={
        "x_range.bounds": (ds.time.min().values, ds.time.max().values)}
    )
def plot_variable_1d(label, label_to_var, ds):
    var = label_to_var[label]
    da = ds[var]
    return da.hvplot(
        x="time",
        title=label,
    ).opts(
    backend_opts={
        "x_range.bounds": (ds.time.min().values, ds.time.max().values)}
    )
def plot_variable_meanstd(label, label_to_var, ds_mean, ds_std):
    var = label_to_var[label]
    da_mean = ds_mean[var]
    da_std= xr.Dataset({"y1":ds_mean[var] - ds_std[var],"y2":ds_mean[var] + ds_std[var]})
    
    band = da_std.hvplot.area(
        x="time", y="y1", y2="y2", alpha=0.3,
        color='gray', label='std',hover=False
    )
    line = da_mean.hvplot(
        x="time",
        title=label,
    )
    return (line * band).opts(
    backend_opts={
        "x_range.bounds": (ds_mean.time.min().values, ds_mean.time.max().values)}
    )
prof = profiles()
capes = cape()
ds_tmser = ds_tmser
crosses = crosses()
prof_plot = hv.DynamicMap(lambda x: plot_variable_heat(x, get_label_to_var(prof), ds1=prof), kdims=[get_variable_dim(prof)])
tsmer_plot = hv.DynamicMap(lambda x: plot_variable_1d(x, get_label_to_var(ds_tmser), ds_tmser), kdims=[get_variable_dim(ds_tmser)])
cape_plot = hv.DynamicMap(lambda x: plot_variable_meanstd(x, get_label_to_var(capes[0]), capes[0], capes[1]), kdims=[get_variable_dim(capes[0])])
cross_plot = hv.DynamicMap(lambda x: plot_variable_meanstd(x, get_label_to_var(crosses[0]), crosses[0], crosses[1]), kdims=[get_variable_dim(crosses[0])])

pl = pn.Card(prof_plot, pn.layout.Divider(), tsmer_plot, pn.layout.Divider(), cape_plot, pn.layout.Divider(),cross_plot, title="Responsive", sizing_mode='stretch_width')
hvplot.show(pl)

Launching server at http://localhost:57370


In [129]:
import holoviews as hv
import hvplot.xarray  # noqa

hv.extension("bokeh")



pl

:DynamicMap   [variable]
   :Overlay
      .Curve.I  :Curve   [time]   (Ground heat flux)
      .Area.Std :Area   [time]   (y1,y2)